# Bộ dữ liệu luận văn · slidespeech-devtest

  1 lpm-videos            244 LPM videos (from thivntrung/dataset-video), checked, train/held-out split
  2 slidespeech-devtest   SlideSpeech dev + test videos and the related files (OCR, transcripts)
  3 slidespeech-train100  100 SlideSpeech train videos, streamed out of the 232 GB archive
  4 vn-test12             12 Vietnamese lecture videos from 5 channel types + SponsorBlock labels
  5 m3av-maths            M3AV Oxford Mathematics: slide OCR/LaTeX labels + the 51 videos


In [ ]:
import csv, glob, json, os, shutil, subprocess, sys, time
from pathlib import Path

def sh(cmd, check=False):
    r = subprocess.run(cmd, capture_output=True, text=True)
    if check and r.returncode:
        raise RuntimeError(r.stderr[-800:])
    return r

def probe(path):
    r = sh(['ffprobe', '-v', 'error', '-print_format', 'json', '-show_streams', '-show_format', str(path)])
    try:
        j = json.loads(r.stdout)
    except Exception:
        return None
    v = [s for s in j.get('streams', []) if s.get('codec_type') == 'video']
    a = [s for s in j.get('streams', []) if s.get('codec_type') == 'audio']
    return {'vcodec': v[0].get('codec_name', '') if v else '', 'width': int(v[0].get('width', 0)) if v else 0,
            'height': int(v[0].get('height', 0)) if v else 0, 'has_audio': bool(a),
            'acodec': a[0].get('codec_name', '') if a else '',
            'duration': round(float(j.get('format', {}).get('duration', 0) or 0), 2)}

def pyav_decodes(path, at=30.0):
    """True if PyAV (what stage 01 uses) can decode a frame of this file."""
    try:
        import av
        with av.open(str(path)) as c:
            s = c.streams.video[0]
            c.seek(int(at / float(s.time_base)), stream=s, backward=True, any_frame=False)
            for frame in c.decode(s):
                frame.to_ndarray(format='rgb24')
                return True
    except Exception as e:
        print('  PyAV lỗi', path.name, str(e)[:120])
    return False

def write_csv(path, rows):
    rows = list(rows)
    with open(path, 'w', newline='', encoding='utf-8') as f:
        w = csv.DictWriter(f, fieldnames=list(rows[0])); w.writeheader(); w.writerows(rows)

def du_gb(path):
    return sum(f.stat().st_size for f in Path(path).rglob('*') if f.is_file()) / 2**30

def install_ytdlp():
    sh([sys.executable, '-m', 'pip', 'install', '-q', '-U', 'yt-dlp[default]'])
    if not shutil.which('deno'):
        os.system('curl -fsSL https://deno.land/install.sh | sh -s -- -y > /dev/null 2>&1')
        os.environ['PATH'] = os.path.expanduser('~/.deno/bin') + ':' + os.environ['PATH']
    print(sh([sys.executable, '-m', 'yt_dlp', '--version']).stdout.strip(), shutil.which('deno'))

BLOCKED_IN_A_ROW = [0]


def youtube(video_id, out, max_height=720, max_sec=None, tries=4, cookies=None, pause=(4, 10)):
    """mp4 with H.264 video (PyAV/OpenCV read it everywhere) and AAC audio; cut to max_sec.
    Uses deno explicitly for YouTube's JS challenge; pauses between videos; after 3 videos in a
    row are refused ("Sign in to confirm you're not a bot") it returns 'blocked' at once."""
    import random
    url = f'https://www.youtube.com/watch?v={video_id}'
    if BLOCKED_IN_A_ROW[0] >= 3:
        return {'status': 'blocked', 'reason': 'dừng: YouTube chặn 3 video liên tiếp'}
    time.sleep(random.uniform(*pause))
    deno = shutil.which('deno') or os.path.expanduser('~/.deno/bin/deno')
    js = ['--js-runtimes', f'deno:{deno}'] if os.path.exists(deno) else []
    formats = [f'bv*[height<={max_height}][vcodec^=avc1]+ba[ext=m4a]',
               f'bv*[height<={max_height}][ext=mp4]+ba[ext=m4a]', f'bv*[height<={max_height}]+ba/b']
    target = Path(out) / f'{video_id}.mp4'
    last = ''
    for attempt in range(tries):
        for f in Path(out).glob(f'{video_id}.*'):
            f.unlink()
        r = sh([sys.executable, '-m', 'yt_dlp', '-q', '--no-warnings', '--ignore-config', *js, '--no-playlist',
                '--socket-timeout', '30', '-f', formats[min(attempt, 2)],
                '--merge-output-format', 'mp4', '--retries', '10', '--fragment-retries', '10',
                *(([('--cookies-from-browser' if cookies[0] == 'browser' else '--cookies'), cookies[1]]) if cookies else []),
                '-o', str(target), url])
        if target.exists():
            info = probe(target)
            if info and info['height'] and info['has_audio']:
                if max_sec and info['duration'] > max_sec + 5:
                    cut = target.with_suffix('.cut.mp4')
                    sh(['ffmpeg', '-v', 'error', '-y', '-i', str(target), '-t', str(max_sec), '-c', 'copy', str(cut)], check=True)
                    original = info['duration']
                    cut.replace(target); info = probe(target); info['original_duration'] = original
                BLOCKED_IN_A_ROW[0] = 0
                return {'status': 'ok', **info}
            last = f'thiếu hình/tiếng: {info}'
        else:
            last = (r.stderr or '').strip().splitlines()[-1:] or ['không có file']
            last = last[0][:200]
            if 'Sign in' in last or 'not a bot' in last:
                BLOCKED_IN_A_ROW[0] += 1
                return {'status': 'blocked', 'reason': last}
        time.sleep(10 * (attempt + 1))
    return {'status': 'failed', 'reason': last}


In [ ]:
OUT = Path('/kaggle/working/slidespeech-devtest'); OUT.mkdir(parents=True, exist_ok=True)
WITH_RELATED = True


In [ ]:
install_ytdlp()
BASE = 'https://speech-lab-share-data.oss-cn-shanghai.aliyuncs.com/SlideSpeech/'
TMP = Path('/kaggle/tmp/ss'); TMP.mkdir(parents=True, exist_ok=True)
# Kho Aliyun đã gỡ các file video nén; còn file nhãn và bảng ánh xạ video -> YouTube (cách tải chính thức).
for name in ('wavid2channel.tar.gz',) + (('related_files.tar.gz',) if WITH_RELATED else ()):
    sh(['curl', '-sS', '-L', '--retry', '5', '-o', str(TMP / name), BASE + name], check=True)
    target = OUT / 'related_files' if name.startswith('related') else TMP / 'map'
    target.mkdir(parents=True, exist_ok=True)
    sh(['tar', '-xzf', str(TMP / name), '-C', str(target)], check=True)
    (TMP / name).unlink()
    print(name, 'xong')
MAP = {}
for f in (TMP / 'map').rglob('wavid2channel.json'):
    MAP[f.parent.name] = json.load(open(f, encoding='utf-8'))
print({k: len(v) for k, v in MAP.items()})


In [ ]:
rows = []
for split in ('dev', 'test'):
    (OUT / 'video' / split).mkdir(parents=True, exist_ok=True)
    for wavid, yt in sorted(MAP.get(split, {}).items()):
        result = youtube(yt, OUT / 'video' / split)
        if result['status'] == 'ok':
            (OUT / 'video' / split / f'{yt}.mp4').rename(OUT / 'video' / split / f'{wavid}.mp4')
        print(split, wavid, result['status'], result.get('duration'), result.get('reason', ''), flush=True)
        rows.append({'wavid': wavid, 'youtube_id': yt, 'category': wavid.split('_')[0], 'split': split,
                     'file': f'video/{split}/{wavid}.mp4' if result['status'] == 'ok' else '',
                     **{k: result.get(k, '') for k in ('status', 'duration', 'height', 'vcodec', 'reason')}})
write_csv(OUT / 'videos.csv', rows)
from collections import Counter
summary = {'videos_ok': sum(r['status'] == 'ok' for r in rows), 'videos': len(rows),
           'hours': round(sum(float(r['duration'] or 0) for r in rows) / 3600, 2),
           'status': dict(Counter(r['status'] for r in rows)), 'size_gb': round(du_gb(OUT), 2)}
(OUT / 'README.md').write_text(f"""# SlideSpeech dev + test

SlideSpeech (Wang et al., ICASSP 2024, https://slidespeech.github.io/), licence CC BY-SA 4.0:
YouTube conference/lecture videos with slides. The video archives were removed from the
authors' storage, so the videos were downloaded from YouTube with the official id mapping
(wavid2channel). Here: dev and test ({summary['videos_ok']}/{summary['videos']} videos, {summary['hours']} h),
H.264 <= 720p + AAC, and `related_files/` (segments, transcriptions, OCR). `videos.csv`: wavid,
YouTube id, category, split, status. Role in the thesis: English test set from unseen channels
(together with 2 held-out LPM speakers).
""", encoding='utf-8')
json.dump(summary, open(OUT / 'summary.json', 'w'), indent=1)
print(summary)
